# YOLO11 객체탐지 — ARIES NPU 추론 데모

ARIES NPU에서 **YOLO11**(11n/11m/11l)로 이미지 → bbox. **모델은 `MXQ` 경로만 바꾸면** 동일 코드로 동작.

- 커널: `pe_npu_host` (qbruntime 설치된 추론 env)
- MXQ 준비: `python -m yolo_npu.compile --model yolo11m --calib <val2017> --calib-num 200 --out ./yolo_out` (`README.md` 2절)
- 정확도(yolo11m, val2017 300장): NPU INT8 mAP@.5:.95 **0.5315** vs fp32 0.5537 (양자화 −4.0%, ~96% 유지)
- 파이프라인: letterbox 640 + RGB + /255 → NPU `(1,8400,84)` → conf필터 + NMS + 좌표역변환

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))  # 레포 루트 (yolo_npu import)
import cv2, matplotlib.pyplot as plt
from yolo_npu import YOLONPU

IMAGE = "../../scratch_yolo/bus.jpg"          # 테스트 이미지 (본인 것으로 교체 가능)

def show(img_bgr, title=None, figsize=(9, 12)):
    plt.figure(figsize=figsize); plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.axis('off');
    if title: plt.title(title)
    plt.show()

## 1. YOLO11m 로드 + 추론 + bbox 시각화

`YOLONPU.load("yolo11m","single")` — **HF에서 자동 다운로드**(없으면 컴파일 안내). 모델을 바꾸려면 `"yolo11n"`/`"yolo11l"`로.

In [ ]:
det = YOLONPU.load("yolo11m", "single", conf_thres=0.25, iou_thres=0.45)  # HF 먼저→없으면 컴파일 안내

boxes = det(IMAGE)
print(f'yolo11m → 검출 {len(boxes)}개:')
for x1, y1, x2, y2, cf, c in boxes:
    print(f'  {det.names[c]:12s} {cf:.2f}  ({int(x1)},{int(y1)})-({int(x2)},{int(y2)})')
show(det.draw(IMAGE, boxes), f'YOLO11m (NPU) — {len(boxes)} objects')

## 2. 모델 스왑 비교 — YOLO11 n / m / l

동일 코드, `mxq`만 교체. n=빠름/가벼움, l=느림/정확.

In [ ]:
variants = ['yolo11n', 'yolo11m', 'yolo11l']
fig, axes = plt.subplots(1, len(variants), figsize=(6 * len(variants), 8))
for ax, name in zip(axes, variants):
    d = YOLONPU.load(name, 'single')     # HF에서 자동 (인자만 교체)
    t = time.perf_counter(); b = d(IMAGE); ms = (time.perf_counter() - t) * 1000
    ax.imshow(cv2.cvtColor(d.draw(IMAGE, b), cv2.COLOR_BGR2RGB))
    ax.set_title(f'{name}: {len(b)} obj, {ms:.0f} ms'); ax.axis('off')
    del d
plt.tight_layout(); plt.show()

## 3. 멀티카드 — 여러 NPU 자동 분산

`device_ids='auto'`(전체) 또는 `[0, 1]`(지정). 배치는 `detect_batch`로 카드에 라운드로빈 분산.
NPU-only 처리량: 1장 198 → 2장 362 → 7장 1072 img/s (yolo11m, B=64).

In [ ]:
from yolo_npu import detect_npu_devices
print('감지된 NPU:', detect_npu_devices())

det_mc = YOLONPU.load("yolo11m", "single", device_ids='auto')  # HF에서 받아 전 카드 분산 (또는 device_ids=[0,1])
print(f'{len(det_mc)}장 사용')

imgs = [IMAGE] * 16
t = time.perf_counter(); results = det_mc.detect_batch(imgs); ms = (time.perf_counter() - t) * 1000
print(f'배치 {len(imgs)}장 → {ms:.0f} ms ({ms/len(imgs):.1f} ms/img), 검출수 {[len(r) for r in results]}')